# figs_team

> Publication figures for teams of Claude agents (guards against team size, the defences against liars, gullibility at scale), drawn from compact per-seed data with one convention: mean ± SEM, and an asterisk at p < 0.05.

In [ ]:
#| default_exp figs_team

In [ ]:
#| hide
from nbdev.showdoc import *

Each figure takes compact data (team sizes, seeds and boards as string keys, as JSON stores them). `tools/figures.py` extracts the data from `results/multiagent/` and saves it to `data/highlights/highlights.json`, so the [highlights](highlights.ipynb) page can redraw the figures without the raw results. Error bars are one standard error of the mean over the independent unit (or the bootstrap standard error, for a ratio measure, where the data carry it). An asterisk marks p < 0.05 in a two-sided test against the stated reference: an exact sign-flip test up to 15 pairs (seeded random flips beyond), or an exact permutation test between team sizes; uncorrected. `guardsScale` also gives each design's trend over team size in its legend: the mean of each seed's least-squares slope over log2 N, per doubling, with a two-sided sign-flip test over seeds (`trend`). `paper=True` draws a figure at the printed width of a NeurIPS page (5.5 in) with the preprints' font sizes and leaves the convention to the caption. With no `path`, a figure is returned for display; with one, it is saved at 200 dpi and closed.

In [ ]:
#| export
import functools
import itertools
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from explRL.bench_stats import signFlipP

ALPHA, NAN, W = 0.05, float('nan'), 5.5               # W: the NeurIPS text width in inches, so preprint figures print at 1:1
NOTE = 'mean ± SEM; * p < 0.05, two-sided exact test (see caption)'
PAPER = {'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 8, 'xtick.labelsize': 7, 'ytick.labelsize': 7,
         'legend.fontsize': 6.5, 'lines.markersize': 4, 'errorbar.capsize': 2, 'lines.linewidth': 1.2}


def sem(x):
    "Standard error of the mean of the finite values of x (nan for fewer than two)."
    x = np.asarray([v for v in x if np.isfinite(v)], float)
    return float(x.std(ddof=1) / np.sqrt(len(x))) if len(x) > 1 else NAN


def signP(d):
    "Two-sided sign-flip p of a paired mean difference: exact up to 15 pairs, else 40,000 seeded random flips."
    return signFlipP(list(d), n=40000) if len(d) else NAN


def permP(a, b, n=20000, seed=0):
    "Two-sided two-sample permutation p of the difference in means: exact when the splits number at most n."
    a, b = np.asarray(a, float), np.asarray(b, float)
    x, k = np.concatenate([a, b]), len(a)
    obs = abs(a.mean() - b.mean())
    if math.comb(len(x), k) <= n:                     # count first: listing every split of a large sample would not fit in memory
        st = [abs(x[list(s)].mean() - np.delete(x, list(s)).mean()) for s in itertools.combinations(range(len(x)), k)]
        return float(np.mean([v >= obs - 1e-12 for v in st]))
    rng = np.random.default_rng(seed)
    st = [abs((p := rng.permutation(x))[:k].mean() - p[k:].mean()) for _ in range(n)]
    return float((1 + sum(v >= obs - 1e-12 for v in st)) / (1 + n))


def star(ax, x, y, p, horizontal=False, color='k', size=12):
    "An asterisk beside a point (horizontal error bars) or above it (vertical) when p < ALPHA."
    if np.isfinite(p) and p < ALPHA:
        if horizontal:
            ax.annotate('*', (x, y), xytext=(4, -5), textcoords='offset points', fontsize=size, color=color, va='center')
        else:
            ax.annotate('*', (x, y), xytext=(0, 1), textcoords='offset points', fontsize=size, color=color, ha='center', va='bottom')


def save(fig, path):
    "Save at 200 dpi with no software stamp, so a redraw is byte-identical, and close."
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=200, metadata={'Software': None})
    plt.close(fig)


def drawn(f):
    "Draw f(d, paper) inside the preprints' font sizes when paper; save to path and close, or return the figure."
    @functools.wraps(f)
    def g(d, path=None, paper=False):
        with plt.rc_context(PAPER if paper else {}):
            fig = f(d, paper)
            if path is not None:
                save(fig, path)
        return fig
    return g

In [ ]:
#| export
SCALE_ARMS = (('colonyGuards', 'guards (one board)', '#1f6391', 'o', '-'), ('squads', 'squads of 8 with guards', '#2a9d8f', 's', '-'),
              ('failFlag', 'guards + failure flags', '#c9a227', '^', '--'), ('guardDeal', 'guards + a copy of every clue', '#b03a2e', 'D', '--'))
GUARD_ARMS = (('plain', 'plain board', '#9aa3ad'), ('confirm', 'instruction only', '#c9a227'), ('colony', 'agreement counts', '#5b8fb9'),
              ('colonyGuards', 'counts + guards', '#1f6391'), ('record', 'oracle track record', '#7a5c99'))


def trend(d):
    "Mean per-seed least-squares slope over log2 N, and its two-sided sign-flip p; d: team size -> seed -> value (seeds at every size)."
    Ns = sorted(int(N) for N, v in d.items() if v)
    seeds = [s for s in d[str(Ns[0])] if all(s in d[str(N)] for N in Ns)] if len(Ns) > 1 else []
    sl = [float(np.polyfit(np.log2(Ns), [d[str(N)][s] for N in Ns], 1)[0]) for s in seeds]
    return (float(np.mean(sl)), signP(sl)) if sl else (NAN, NAN)


@drawn
def guardsScale(d, paper):
    """Advantage over the plain board against team size, each design's trend over log2 N in the legend;
    d: arm -> team size -> seed -> difference (arm minus plain board, paired on seed)."""
    fig, ax = plt.subplots(figsize=(W, 2.8) if paper else (7.2, 4.4))
    for k, (arm, lab, col, mk, ls) in enumerate(SCALE_ARMS):
        a = {N: v for N, v in d.get(arm, {}).items() if v}
        if not a:
            continue
        pts = sorted((int(N), list(v.values())) for N, v in a.items())
        x = np.log2([N for N, _ in pts]) + (k - 1.5) * 0.05
        y, e = np.array([np.mean(v) for _, v in pts]), np.array([sem(v) for _, v in pts])
        b, p = trend(a)
        ax.errorbar(x, y, yerr=e, color=col, marker=mk, ls=ls, lw=1.3 if paper else 1.8, ms=4 if paper else 6, capsize=2 if paper else 3,
                    label=f'{lab}: {b:+.3f} per doubling (p = {p:.3f})' if np.isfinite(b) else lab)
        for xi, yi, ei, (_, v) in zip(x, y, e, pts):
            star(ax, xi, yi + ei, signP(v), color=col, size=9 if paper else 12)
    ax.set_xticks(np.log2([8, 16, 32, 64]), ['8', '16', '32', '64'])
    ax.set(xlabel='agents (a quarter of them scripted liars)', ylabel='advantage over the plain board\n(honest pooled share)', ylim=(0, 0.5))
    ax.axhline(0, color='#9aa3ad', lw=0.8)
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(frameon=False, fontsize=None if paper else 8, loc='upper left')
    if not paper:
        ax.set_title('Squads with their own guards hold their advantage as teams grow', fontsize=10)
        ax.text(0.99, 0.01, '8 seeds per point; ' + NOTE + ': against 0, paired on seed\nlegend: mean per-seed slope over log2(agents) '
                'and its two-sided exact sign-flip p', transform=ax.transAxes, ha='right', fontsize=7, color='#566372')
    fig.tight_layout()
    return fig


@drawn
def guards(d, paper):
    "Honest pooled share and answers built on lies per arm and team size; d: metric -> arm -> team size -> seed -> value."
    sizes = sorted({int(N) for arm in d['honestPooled'].values() for N in arm})
    fig, axs = plt.subplots(1, 2, figsize=(W, 2.5) if paper else (12, 4.1))
    w = 0.16
    for ax, k, title, ylab in ((axs[0], 'honestPooled', 'Guards protect the team best', 'honest pooled share (liars present)'),
                               (axs[1], 'contamination', 'and cut the answers built on lies', 'honest answers built on a false clue')):
        for i, (arm, lab, col) in enumerate(GUARD_ARMS):
            xs, ys, es, ps = [], [], [], []
            for j, N in enumerate(sizes):
                a, b = d[k].get(arm, {}).get(str(N)), d[k]['plain'].get(str(N), {})
                if not a:
                    continue
                xs.append(j + (i - 2) * w)
                ys.append(np.mean(list(a.values())))
                es.append(sem(list(a.values())))
                ps.append(signP([a[s] - b[s] for s in sorted(a, key=int) if s in b]) if arm != 'plain' else NAN)
            ax.bar(xs, ys, w * 0.92, yerr=es, capsize=1.5 if paper else 2, color=col, label=lab, error_kw=dict(lw=0.8) if paper else None)
            for x, y, e, p in zip(xs, ys, es, ps):
                star(ax, x, max(y, 0) + e, p, size=8 if paper else 11)
        ax.set_xticks(range(len(sizes)), [f'{N} agents' for N in sizes])
        ax.set(title='' if paper else title, ylabel=ylab.replace('honest answers', 'answers') if paper else ylab)
        ax.axhline(0, color='k', lw=0.6)
    axs[0].set_ylim(-0.13, 0.38 if paper else 0.45)
    axs[1].set_ylim(0, 1.0)
    if paper:
        fig.legend(*axs[0].get_legend_handles_labels(), loc='upper center', ncol=5, frameon=False, handlelength=1.2, columnspacing=1.0)
        fig.tight_layout(rect=(0, 0, 1, 0.92))
    else:
        axs[0].legend(fontsize=8, loc='upper center', ncol=3)
        fig.text(0.5, 0.005, '8 seeds per bar; ' + NOTE + ': each arm against the plain board, paired on seed (at 32 agents only the '
                 'plain board and guards ran)', ha='center', fontsize=8, color='#566372')
        fig.tight_layout(rect=(0, 0.03, 1, 1))
    return fig

In [ ]:
#| export
@drawn
def trustScale(d, paper):
    """Left: the track record's damage recovered and the request board's gap closed against team size (rows [N, gap, se, p]).
    Right: honest answers built on a false clue per runner and team size (per-seed values; permutation test against the smallest team)."""
    fig, (a, b) = plt.subplots(1, 2, figsize=(W, 2.4) if paper else (11, 4.0))
    sz = 9 if paper else 12
    for key, col, lab in (('record', '#1f6391', 'track record (each sender\'s past accuracy shown): damage won back'),
                          ('request', '#c0392b', 'request board (agents can ask for clues): sharing gap closed')):
        pts = d[key]
        x, y, e = [r[0] for r in pts], np.array([r[1] for r in pts]), np.array([r[2] for r in pts])
        a.errorbar(x, y, yerr=e, fmt='o-', capsize=2 if paper else 3, color=col, label=lab.split(':')[0] if paper else lab)
        for xi, yi, ei, r in zip(x, y, e, pts):
            star(a, xi, yi + ei if yi >= 0 else yi - ei - 0.05, r[3], color=col, size=sz)
    a.axhline(0, color='k', lw=0.6)
    a.set(xscale='log', xticks=[4, 8, 16, 32], xlabel='team size (Claude Haiku 5.5 agents)', ylim=(-0.32, 0.55 if paper else 0.7),
          ylabel='damage recovered or gap closed' if paper else 'share of the gap closed (1 = all of it)',
          title='' if paper else 'Two fixes keep their effect from 4 to 32 agents')
    a.set_xticklabels(['4', '8', '16', '32']), a.legend(fontsize=None if paper else 8, loc='center left' if paper else 'upper right')
    for who, mk, col, lab in (('haiku', 'o-', '#1f6391', 'Claude Haiku 5.5'), ('numpy', 's--', '#999999', 'rule agents')):
        per = {int(N): v for N, v in d['contaminated'][who].items() if v}
        Ns = sorted(per)
        y, e = np.array([np.mean(per[N]) for N in Ns]), np.array([sem(per[N]) for N in Ns])
        b.errorbar(Ns, y, yerr=e, fmt=mk, capsize=2 if paper else 3, color=col, label=lab)
        for N, yi, ei in zip(Ns[1:], y[1:], e[1:]):
            star(b, N, yi + ei, permP(per[N], per[Ns[0]]), color=col, size=sz)
    b.set(xscale='log', xticks=[8, 16, 32], xlabel='team size (a quarter of the agents liars)', ylim=(0.6, 1.04),
          ylabel='answers built on a false clue' if paper else 'honest answers built on a false clue',
          title='' if paper else 'Larger teams build on lies more often')
    b.set_xticklabels(['8', '16', '32'])
    b.legend(fontsize=None if paper else 8, loc='lower right')
    for ax in (a, b):
        ax.minorticks_off()
    if paper:
        fig.tight_layout()
    else:
        fig.text(0.5, 0.005, 'Left: 10 seeds at 4 agents, 5 per larger size; mean ± bootstrap SE; * p < 0.05, two-sided exact sign-flip test '
                 'against 0\n(with 5 seeds the exact two-sided p cannot fall below 0.0625). Right: mean ± SEM over seeds (5 for Claude '
                 'Haiku, 20 for rule agents); * p < 0.05, exact permutation test against 8 agents.', ha='center', fontsize=7, color='#566372')
        fig.tight_layout(rect=(0, 0.07, 1, 1))
    return fig

## Tests

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
from explRL.figs_team import *

# the helpers: SEM ignores nan; the sign-flip p of 8 all-positive pairs is the exact two-sided 2/256; permutation p is exact for small samples
assert abs(sem([1.0, 2.0, 3.0, float('nan')]) - 1 / np.sqrt(3)) < 1e-12 and np.isnan(sem([1.0]))
assert abs(signP([0.1] * 8) - 2 / 256) < 1e-12 and np.isnan(signP([]))
assert abs(permP([0.0, 0.0, 0.0], [1.0, 1.0, 1.0]) - 2 / 20) < 1e-12 and permP([0.5, 0.6], [0.5, 0.6]) == 1.0

# every figure draws from compact data with string keys, at web and printed size, and saves byte-identical files
seeds = [str(s) for s in range(8)]
gs = {'colonyGuards': {'8': {s: 0.2 + 0.01 * int(s) for s in seeds}, '64': {s: 0.1 + 0.01 * int(s) for s in seeds}},
      'squads': {'16': {s: 0.3 for s in seeds}}}
g = {k: {a: {N: {s: 0.1 * j + 0.01 * int(s) for s in seeds} for N in ('8', '16')} for j, (a, *_) in enumerate(GUARD_ARMS)}
     for k in ('honestPooled', 'contamination')}
t = dict(record=[[4, 0.3, 0.05, 0.004], [8, 0.3, 0.06, 0.06]], request=[[8, -0.16, 0.02, 0.06]],
         contaminated={'haiku': {'8': [0.78, 0.8, 0.79], '16': [0.9, 0.88, 0.89]}, 'numpy': {'8': [0.97] * 3, '16': [0.98] * 3}})
with tempfile.TemporaryDirectory() as tmp:
    for f, d in ((guardsScale, gs), (guards, g), (trustScale, t)):
        for paper in (False, True):
            p1, p2 = Path(tmp) / f'{f.__name__}{paper}1.png', Path(tmp) / f'{f.__name__}{paper}2.png'
            f(d, p1, paper=paper), f(d, p2, paper=paper)
            assert p1.stat().st_size > 1000 and p1.read_bytes() == p2.read_bytes()
fig = guardsScale(gs)                                   # no path: the figure comes back for display
assert len(fig.axes) == 1 and len(fig.axes[0].texts) == 4   # three significant points starred, and the footnote
# the trend: every seed falls by 0.1 over three doublings (exact two-sided p 2/256); one size gives no trend
b, p = trend(gs['colonyGuards'])
assert abs(b + 0.1 / 3) < 1e-12 and abs(p - 2 / 256) < 1e-12 and all(np.isnan(trend(gs['squads'])))
labels = [t.get_text() for t in fig.axes[0].get_legend().get_texts()]
assert labels == ['guards (one board): -0.033 per doubling (p = 0.008)', 'squads of 8 with guards']

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()